# 02 — Sessions and Dataset (Ahmed, S1.8)

## Plan

1. Load the project configuration and selected sessions.
2. Describe the frozen IBL Brain-Wide Map release and the documented session-selection criteria.
3. Report the 10 selected sessions and their neuron/region coverage using `sessions.yaml`.
4. Explain how ready sessions are built, and inspect the shape of any **actually generated** `.npz` files.
5. Record what remains unverified, without inventing dataset counts or model results.

**Source of truth:** `config.yaml`, `sessions.yaml`, shared project helpers, and any generated `data/processed/*.npz`. Code is not reimplemented in this notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the brain-decoding repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.settings import (
    load_config, load_session_list, load_session_info,
    list_processed_sessions, load_session,
)

import pandas as pd
import numpy as np
import yaml

config = load_config()
print("Project:", config["project_name"])
print("Release freeze:", config["ibl"]["release_freeze"])
print("Aggregate tag:", config["ibl"]["aggregate_tag"])

## 1. Session selection criteria

The project uses the IBL Brain-Wide Map freeze specified in `config.yaml`. According to the team's S1.2 plan, eligible sessions must satisfy the official trial filter, good-neuron quality rules, Beryl region coverage, one session per mouse and the per-lab limit. The code selects the highest-neuron eligible sessions first, with reserve sessions for replacement.

The summary below reads the **actual configuration**; it is not a claim that every criterion has independently been audited.

In [ ]:
d = config["data"]
criteria = {
    "Target sessions": d["n_sessions"],
    "Minimum eligible neurons": d["min_neurons"],
    "Minimum eligible trials": d["min_trials"],
    "Minimum error trials": d["min_errors"],
    "Minimum qualifying brain regions": d["min_regions"],
    "Minimum neurons per qualifying region": d["neurons_per_region"],
    "Maximum sessions per lab": d["max_sessions_per_lab"],
    "Good neuron QC": d["good_neuron_qc"],
    "Region mapping": d["region_mapping"],
    "Excluded regions": ", ".join(d["excluded_regions"]),
    "Feature window (seconds)": str(d["window"]),
}
display(pd.DataFrame(criteria.items(), columns=["Selection criterion", "Configuration value"]))

## 2. Selected sessions and reserves

`src/data/sessions.py` owns the selection logic and writes `sessions.yaml`. The next cells inspect that file without downloading data or modifying it.

In [ ]:
with (ROOT / "sessions.yaml").open(encoding="utf-8") as f:
    selection = yaml.safe_load(f)
selected = selection.get("sessions", [])
reserves = selection.get("reserves", [])

rows = []
for s in selected:
    rows.append({
        "session_id": s["session_id"],
        "subject": s["subject"],
        "lab": s["lab"],
        "n_pids": len(s["pids"]),
        "n_neurons": s["n_neurons"],
        "n_regions": s["n_regions"],
    })
summary = pd.DataFrame(rows)
print("Selected sessions:", len(selected))
print("Reserve sessions:", len(reserves))
display(summary)

In [ ]:
# Checks available from the saved selection metadata.
assert len(selected) == d["n_sessions"], "Wrong number of selected sessions"
assert summary["subject"].nunique() == len(selected), "Repeated subject"
assert (summary["lab"].value_counts() <= d["max_sessions_per_lab"]).all(), "Lab limit exceeded"
assert (summary["n_neurons"] >= d["min_neurons"]).all(), "Neuron minimum not met"
assert (summary["n_regions"] >= d["min_regions"]).all(), "Region count minimum not met"
assert all(len(s["pids"]) > 0 for s in selected), "A session has no probe IDs"
print("PASS: saved session count, subject uniqueness, lab limits and recorded neuron/region counts")
print("Note: trial counts and qualifying-region neuron counts need raw IBL metadata verification.")

## 3. Filtering / release size audit

The selection script reads `bwm_query(freeze=...)`, applies official trial eligibility via `filter_sessions`, then applies good-neuron/region thresholds and subject/lab constraints. **The current `sessions.yaml` contains selected sessions and reserves, but not the counts rejected by each selection rule.** Those intermediate audit counts cannot be reconstructed from selected sessions alone.

The following displays the number of sessions in the frozen release **only when live IBL access succeeds**; it does not invent intermediate filtering counts.

In [ ]:
try:
    from brainwidemap import bwm_query
    release_sessions = bwm_query(freeze=config["ibl"]["release_freeze"])
    print("Distinct sessions in release:", release_sessions["eid"].nunique())
except Exception as exc:
    print("Release-size query unavailable:", type(exc).__name__, str(exc)[:180])

print("Selected:", len(selected), "| Reserves:", len(reserves))
print("Excluded per-rule counts: not recorded in sessions.yaml; require an S1.2 selection audit.")

## 4. How one ready session is built

This is the implementation sequence specified by the team plan:

`load_session_info` → `load_trials` → `clean_trials` → `load_neurons` → window around `firstMovement_times` → `firing_rates` → `assemble_session` → `save_session`.

Each row of `X` is a trial; each column is a good neuron. Features are firing rates in Hz in the interval **[−0.1 s, 0.0 s] relative to first wheel movement**. The selected choice is `y=1` for `choice=+1` and `y=0` for `choice=−1`.

`build_dataset.py` is responsible for executing this process; the notebook only describes and inspects it. Its execution requires teammate-owned `clean_trials.py`, `neurons.py`, and `features.py` to be present and compatible.

In [ ]:
expected_keys = {
    "X", "y", "trial_number", "stim_side", "contrast", "is_error",
    "is_zero_contrast", "prob_left", "reaction_time", "neuron_id",
    "neuron_region", "session_id",
}
ready = list_processed_sessions()
print("Processed sessions actually available:", len(ready), "of", len(selected))

processed_rows = []
for session_id in ready:
    data = load_session(session_id)
    keys = set(data)
    if keys != expected_keys:
        raise AssertionError(f"Unexpected keys for {session_id}: missing={expected_keys - keys}, extra={keys - expected_keys}")
    x = np.asarray(data["X"])
    assert x.ndim == 2 and np.isfinite(x).all(), f"Invalid X: {session_id}"
    assert x.shape[0] == len(data["y"]) and x.shape[1] == len(data["neuron_id"])
    processed_rows.append({
        "session_id": session_id,
        "n_trials": x.shape[0],
        "n_neurons": x.shape[1],
        "n_regions": len(set(map(str, data["neuron_region"]))),
        "X_shape": str(x.shape),
    })

if processed_rows:
    display(pd.DataFrame(processed_rows))
else:
    print("No ready .npz files found. Shape tables will populate automatically after S1.7 succeeds.")

## Conclusions

- The selected-session list, per-session metadata and reserve list are read directly from `sessions.yaml`.
- Basic selection metadata checks are executed in this notebook; eligibility details that require raw tables must be checked against IBL data.
- No `.npz` file or quantitative result is fabricated. The dataset-shape table appears only for **real processed files**.
- Completion of S1.7 still depends on the teammate modules and a full real-data run. Once those files are available, rerun this notebook using **Kernel → Restart & Run All** and update the selection audit if needed.

**Status:** documentation and checks ready; processed-session results pending the S1.7 integration test.